<!-- notebook-header -->
# Classificacao de Imagens: Pipeline Completo

**Modulo:** 05 - Dominios Aplicados / 05A - Computer Vision  
**Tipo:** Aula com exercicios guiados e solucoes executaveis  
**Descricao:** Dados de imagem, augmentations, desbalanceamento, avaliacao e interpretabilidade.


# Classificacao de Imagens: Pipeline Completo

**Modulo 5A -- Visao Computacional | Notebook 2 de 5**

Neste notebook, construiremos um pipeline completo de classificacao de imagens:
desde transfer learning ate avaliacao com metricas adequadas. Exploraremos tecnicas
avancadas como discriminative learning rates, loss functions especializadas, e
test-time augmentation (TTA).

## Pre-requisitos e Fio Narrativo

**Antes deste notebook voce deve ter estudado:**
- CNNs fundamentos (5A_1) -- convolucao, pooling, arquiteturas
- Transfer learning (4_4) -- conceito de reusar pesos pre-treinados
- Treinamento deep (4_3) -- learning rate scheduling, early stopping

**Fio narrativo:** Em 5A_1, entendemos *como* CNNs funcionam. Agora vamos aprender
*como usa-las na pratica*: escolher a estrategia de transfer learning certa, configurar
o pipeline de treino com tecnicas modernas, avaliar com metricas adequadas, e resolver
problemas comuns como class imbalance e overfitting.

**O que vamos construir:** Um toolkit mental para classificacao de imagens -- desde
"tenho um dataset, qual estrategia usar?" ate "como interpretar os resultados?".

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    import torchvision.models as models
    HAS_TORCH = True
    print(f'PyTorch version: {torch.__version__}')
except ImportError:
    HAS_TORCH = False
    print('PyTorch nao disponivel -- demonstracoes conceituais com NumPy')

np.random.seed(42)
print('Imports carregados com sucesso!')

## 1. Transfer Learning para Classificacao

### Analogia: Medico Especialista vs Generalista

Imagine treinar um medico do zero: anos de estudo geral + especializacao.
Transfer learning e como pegar um medico generalista ja formado e dar apenas
o treinamento de especializacao:
- **Feature Extraction** = manter toda a formacao geral, treinar so a especialidade
- **Fine-tuning** = ajustar tambem parte da formacao geral para a nova area
- **Discriminative LR** = ajustar mais a especialidade, menos a base geral

### Definicao Formal

Transfer learning reutiliza pesos de um modelo treinado em um dataset grande (ImageNet)
para uma tarefa nova. Tres estrategias principais:

1. **Feature Extraction:** congela backbone, treina apenas o classifier final
2. **Fine-tuning:** descongela gradualmente, treina tudo com LR pequeno
3. **Discriminative LRs:** LRs diferentes por camada (menores no inicio, maiores no fim)

### Por que em ML: O Paradigma Dominante

Na pratica, quase ninguem treina CNNs do zero. ImageNet pre-training fornece features
genericas (bordas, texturas, formas) que transferem para quase qualquer tarefa visual.
Isso reduz dados necessarios de milhoes para milhares de imagens.

In [ ]:
# Visualizacao: Feature Extraction vs Fine-tuning vs Discriminative LR
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

layers = ['Conv1', 'Conv2', 'Conv3', 'Conv4', 'Conv5', 'FC']
n = len(layers)

# Feature Extraction
ax = axes[0]
colors = ['#3498db'] * (n-1) + ['#e74c3c']
frozen = [True] * (n-1) + [False]
for i, (layer, color, frz) in enumerate(zip(layers, colors, frozen)):
    alpha = 0.3 if frz else 1.0
    ax.barh(i, 1, color=color, alpha=alpha, edgecolor='black')
    status = 'CONGELADO' if frz else 'TREINA'
    ax.text(0.5, i, f'{layer}: {status}', ha='center', va='center',
            fontweight='bold', fontsize=9)
ax.set_title('Feature Extraction', fontsize=12, fontweight='bold')
ax.set_xlim(0, 1)
ax.axis('off')

# Fine-tuning
ax = axes[1]
colors_ft = ['#3498db'] * 3 + ['#e74c3c'] * 3
frozen_ft = [True, True, True, False, False, False]
for i, (layer, color, frz) in enumerate(zip(layers, colors_ft, frozen_ft)):
    alpha = 0.3 if frz else 1.0
    ax.barh(i, 1, color=color, alpha=alpha, edgecolor='black')
    status = 'CONGELADO' if frz else 'TREINA'
    ax.text(0.5, i, f'{layer}: {status}', ha='center', va='center',
            fontweight='bold', fontsize=9)
ax.set_title('Fine-tuning (parcial)', fontsize=12, fontweight='bold')
ax.set_xlim(0, 1)
ax.axis('off')

# Discriminative LR
ax = axes[2]
lrs = [0.00001, 0.0001, 0.001, 0.005, 0.01, 0.05]
max_lr = max(lrs)
for i, (layer, lr) in enumerate(zip(layers, lrs)):
    width = lr / max_lr
    ax.barh(i, width, color='#2ecc71', alpha=0.8, edgecolor='black')
    ax.text(width + 0.02, i, f'{layer}: LR={lr}', va='center', fontsize=9, fontweight='bold')
ax.set_title('Discriminative LR', fontsize=12, fontweight='bold')
ax.set_xlim(0, 1.3)
ax.axis('off')

plt.suptitle('Tres Estrategias de Transfer Learning', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/transfer_strategies.png', dpi=100, bbox_inches='tight')
plt.show()

# Tabela de decisao
print('Quando usar cada estrategia:')
print(f'{"Estrategia":<25} {"Dataset":<15} {"Tempo":<10} {"Accuracy":<10}')
print('-' * 60)
print(f'{"Feature Extraction":<25} {"< 1K imgs":<15} {"Rapido":<10} {"Boa":<10}')
print(f'{"Fine-tuning":<25} {"1K-100K":<15} {"Medio":<10} {"Melhor":<10}')
print(f'{"Discriminative LR":<25} {"> 10K":<15} {"Medio":<10} {"Otima":<10}')
print(f'{"From Scratch":<25} {"> 1M":<15} {"Lento":<10} {"Variavel":<10}')

### O que observar

1. Feature Extraction treina apenas 1 camada -- rapido mas menos flexivel
2. Fine-tuning descongela camadas finais -- as que aprendem features task-specific
3. Discriminative LR treina TUDO, mas com LRs proporcionais a distancia do input
4. Camadas iniciais (bordas, texturas) precisam de menos ajuste que camadas finais
5. A escolha depende do tamanho do dataset e similaridade com ImageNet

### O que concluir

Para a maioria dos projetos reais, **fine-tuning com discriminative LR** e a melhor opcao.
Feature extraction e ideal quando ha pouquissimos dados ou tempo muito limitado.
Treinar do zero so faz sentido com datasets muito grandes (>1M) ou dominios radicalmente
diferentes de ImageNet (ex: imagens satelite, microscopia).

### Conexao com outros notebooks

Em 4_4 (transfer learning), vimos o conceito teorico. Aqui, aplicamos na pratica com
estrategias especificas. A ideia de "congelar" camadas conecta com 4_3 (treinamento),
onde vimos que learning rate afeta convergencia.

## 2. Training Recipes Modernos

### Analogia: Receita de Bolo Profissional

Um padeiro amador segue a receita basica. Um profissional sabe *truques*: pre-aquecer
o forno exatamente, adicionar ingredientes na ordem certa, deixar descansar. Training
recipes sao esses truques para treinar CNNs de forma otima.

### Definicao Formal

Um "training recipe" e um conjunto de tecnicas de treino combinadas:
- **Progressive Resizing:** comecar com imagens pequenas, aumentar gradualmente
- **Test-Time Augmentation (TTA):** aplicar augmentations na inferencia e agregar
- **Model Ensembling:** combinar predicoes de multiplos modelos
- **Label Smoothing:** suavizar targets (0,1 -> 0.05, 0.95)
- **Mixup/CutMix:** combinar imagens e labels durante treino

### Por que em ML: Cada Truque Adiciona 0.5-2% de Accuracy

Individualmente, cada tecnica melhora pouco. Combinadas, podem melhorar 5-10%.
Em competicoes (Kaggle), a diferenca entre 1o e 10o lugar e frequentemente
a qualidade do training recipe, nao a arquitetura.

In [ ]:
# Simulacao: Progressive Resizing
sizes = [64, 128, 224, 320]
epochs_per_size = 5

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Schedule de tamanho
epochs = list(range(len(sizes) * epochs_per_size))
img_sizes = []
for s in sizes:
    img_sizes.extend([s] * epochs_per_size)

ax = axes[0]
ax.step(epochs, img_sizes, where='post', linewidth=2, color='#3498db')
ax.fill_between(epochs, img_sizes, alpha=0.2, step='post', color='#3498db')
ax.set_xlabel('Epoch')
ax.set_ylabel('Tamanho da Imagem (px)')
ax.set_title('Progressive Resizing Schedule', fontweight='bold')
ax.grid(True, alpha=0.3)
for i, s in enumerate(sizes):
    ax.annotate(f'{s}x{s}', xy=(i*epochs_per_size + 2, s),
                fontsize=11, fontweight='bold', color='#2c3e50')

# Simulacao TTA: accuracy vs num augmentations
n_augs = [1, 2, 3, 5, 10, 15, 20]
base_acc = 85.0
tta_accs = [base_acc + 3.0 * (1 - np.exp(-n/5)) for n in n_augs]
tta_times = [0.05 * n for n in n_augs]

ax = axes[1]
ax.plot(n_augs, tta_accs, 'o-', color='#e74c3c', linewidth=2, markersize=8, label='Accuracy')
ax.set_xlabel('Numero de Augmentations (TTA)')
ax.set_ylabel('Test Accuracy (%)', color='#e74c3c')
ax.tick_params(axis='y', labelcolor='#e74c3c')

ax2 = ax.twinx()
ax2.plot(n_augs, tta_times, 's--', color='#2ecc71', linewidth=2, markersize=8, label='Tempo')
ax2.set_ylabel('Tempo por Imagem (s)', color='#2ecc71')
ax2.tick_params(axis='y', labelcolor='#2ecc71')

ax.set_title('TTA: Accuracy vs Tempo', fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/training_recipes.png', dpi=100, bbox_inches='tight')
plt.show()

print('Progressive Resizing: comeca rapido com imagens menores, refina com maiores')
print(f'TTA com 5 augmentations: +{tta_accs[3]-base_acc:.1f}% accuracy, {tta_times[3]:.2f}s extra')

### O que observar

1. Progressive resizing começa com 64x64 (rapido) e termina com 320x320 (refinado)
2. Imagens menores = batches maiores = treinamento mais rapido nas primeiras epocas
3. TTA com 5 augmentations ja captura a maior parte do ganho (~2.5% extra)
4. Apos 10 augmentations, o ganho marginal diminui mas o tempo continua aumentando
5. O ponto otimo e tipicamente 5-10 augmentations para TTA

### O que concluir

Training recipes nao sao "truques magicos" -- cada um resolve um problema especifico.
Progressive resizing acelera convergencia. TTA melhora robustez na inferencia.
Ensembling combina visoes complementares. Na pratica, comece com o basico (augmentation +
LR scheduling) e adicione tecnicas conforme necessario.

### Conexao com outros notebooks

Progressive resizing conecta com 4_5 (aceleracao hardware) -- imagens menores usam
menos memoria GPU. TTA conecta com 5A_1 (data augmentation) -- as mesmas transformacoes
usadas no treino sao reutilizadas na inferencia.

## 3. Loss Functions Avancadas

### Analogia: Sistema de Notas Escolar

- **Cross-Entropy:** nota binaria (certo/errado) -- simples mas ignora nuances
- **Label Smoothing:** nota com margem (0.9 certo, 0.1 duvida) -- mais realista
- **Focal Loss:** professor que foca nos alunos com mais dificuldade
- **Mixup:** "misturar" exercicios para ensinar a generalizar

### Definicao Formal

- **Cross-Entropy:** L = -sum(y * log(p)) -- loss padrao para classificacao
- **Label Smoothing:** targets suavizados: y_smooth = (1-eps)*y + eps/K
- **Focal Loss:** FL(p) = -alpha * (1-p)^gamma * log(p) -- penaliza exemplos faceis menos
- **Mixup:** x_mix = lambda*x1 + (1-lambda)*x2, y_mix = lambda*y1 + (1-lambda)*y2

### Por que em ML: Cada Loss Resolve um Problema Diferente

Cross-Entropy assume classes balanceadas e targets perfeitos.
Focal Loss resolve class imbalance sem alterar o dataset.
Label Smoothing reduz overconfidence do modelo.
Mixup funciona como regularizacao entre classes.

In [ ]:
# Implementacao e visualizacao de loss functions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Cross-Entropy vs Focal Loss
p = np.linspace(0.01, 0.99, 100)  # probabilidade da classe correta

ce_loss = -np.log(p)  # Cross-Entropy
focal_g1 = -(1-p)**1 * np.log(p)   # Focal gamma=1
focal_g2 = -(1-p)**2 * np.log(p)   # Focal gamma=2
focal_g5 = -(1-p)**5 * np.log(p)   # Focal gamma=5

ax = axes[0, 0]
ax.plot(p, ce_loss, 'b-', linewidth=2, label='CE (gamma=0)')
ax.plot(p, focal_g1, 'g--', linewidth=2, label='Focal (gamma=1)')
ax.plot(p, focal_g2, 'r-.', linewidth=2, label='Focal (gamma=2)')
ax.plot(p, focal_g5, 'm:', linewidth=2, label='Focal (gamma=5)')
ax.set_xlabel('Probabilidade da classe correta (p)')
ax.set_ylabel('Loss')
ax.set_title('Cross-Entropy vs Focal Loss', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1)
ax.set_ylim(0, 5)

# 2. Label Smoothing
ax = axes[0, 1]
n_classes = 5
target_class = 2

# Hard labels
hard = np.zeros(n_classes)
hard[target_class] = 1.0

# Smoothed labels (epsilon = 0.1)
eps = 0.1
smooth = np.full(n_classes, eps / n_classes)
smooth[target_class] = 1 - eps + eps / n_classes

x_pos = np.arange(n_classes)
width = 0.35
bars1 = ax.bar(x_pos - width/2, hard, width, label='Hard Labels', color='#3498db')
bars2 = ax.bar(x_pos + width/2, smooth, width, label=f'Smooth (eps={eps})', color='#e74c3c')
ax.set_xlabel('Classe')
ax.set_ylabel('Probabilidade Target')
ax.set_title('Label Smoothing', fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels([f'C{i}' for i in range(n_classes)])
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# 3. Mixup visualization
ax = axes[1, 0]
np.random.seed(42)
# Duas "imagens" simplificadas
img1 = np.random.rand(4, 4) * 0.5 + 0.5  # clara
img2 = np.random.rand(4, 4) * 0.5         # escura
lam = 0.7
mixed = lam * img1 + (1 - lam) * img2

for i, (title, data) in enumerate([
    (f'Img1 (gato)', img1),
    (f'Mixed (lam={lam})', mixed),
    (f'Img2 (cao)', img2),
]):
    sub_ax = fig.add_axes([0.05 + i*0.14, 0.12, 0.12, 0.3])
    sub_ax.imshow(data, cmap='gray', vmin=0, vmax=1)
    sub_ax.set_title(title, fontsize=9)
    sub_ax.axis('off')

ax.text(0.5, 0.7, f'Mixup: lambda = {lam}', ha='center', fontsize=14, fontweight='bold',
        transform=ax.transAxes)
ax.text(0.5, 0.5, f'x_mix = {lam}*gato + {1-lam:.1f}*cao', ha='center', fontsize=12,
        transform=ax.transAxes)
ax.text(0.5, 0.3, f'y_mix = {lam}*[1,0] + {1-lam:.1f}*[0,1] = [{lam:.1f}, {1-lam:.1f}]',
        ha='center', fontsize=12, transform=ax.transAxes)
ax.axis('off')
ax.set_title('Mixup: Combinar Imagens e Labels', fontweight='bold')

# 4. Efeito de Focal gamma em class imbalance
ax = axes[1, 1]
# Simular: 90% classe 0 (facil), 10% classe 1 (dificil)
n_samples = 1000
class_0 = int(0.9 * n_samples)
class_1 = n_samples - class_0

# Probabilidades tipicas
p_easy = np.random.beta(8, 2, class_0)   # alta confianca
p_hard = np.random.beta(2, 5, class_1)   # baixa confianca

for gamma, color, style in [(0, '#3498db', '-'), (2, '#e74c3c', '--'), (5, '#2ecc71', ':')]:
    loss_easy = -(1-p_easy)**gamma * np.log(p_easy + 1e-8)
    loss_hard = -(1-p_hard)**gamma * np.log(p_hard + 1e-8)
    ratio = np.mean(loss_hard) / (np.mean(loss_easy) + 1e-8)
    ax.bar(['gamma=' + str(gamma)], [ratio], color=color, alpha=0.7, edgecolor='black')

ax.set_ylabel('Ratio Loss(dificil) / Loss(facil)')
ax.set_title('Focal Loss: Rebalanceamento', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('/tmp/loss_functions.png', dpi=100, bbox_inches='tight')
plt.show()

print('Focal Loss com gamma=5 foca ~8x mais nos exemplos dificeis')
print('Label Smoothing evita que o modelo fique overconfident (100% -> 90%)')
print('Mixup cria amostras "entre classes", melhorando generalizacao')

### O que observar

1. Focal Loss com gamma alto **reduz drasticamente** a loss de exemplos faceis (p alto)
2. Com gamma=2, exemplos com p=0.9 tem loss 100x menor que com CE padrao
3. Label Smoothing distribui probabilidade para todas as classes -- evita overconfidence
4. Mixup cria targets "suaves" -- o modelo aprende que fronteiras entre classes sao graduais
5. O ratio loss(dificil)/loss(facil) aumenta com gamma, focando o treino em exemplos problematicos

### O que concluir

A escolha de loss function depende do problema:
- **Dataset balanceado:** Cross-Entropy padrao funciona bem
- **Class imbalance:** Focal Loss ou class weights
- **Overconfidence:** Label Smoothing (eps=0.1 e um bom default)
- **Pouco dado:** Mixup/CutMix como regularizacao

### Conexao com outros notebooks

Em 1_2 (estatistica inferencial), vimos probabilidades e distribuicoes. Loss functions
sao essencialmente **criterios probabilisticos** de quao bem o modelo estima as probabilidades
verdadeiras. Focal Loss conecta com 3_1 (classificacao), onde vimos metricas para
classes desbalanceadas.

## 4. Metricas e Avaliacao

### Analogia: Avaliacao de Aluno por Multiplos Criterios

Avaliar um modelo apenas por accuracy e como avaliar um aluno apenas pela nota final.
Precisamos de multiplas perspectivas: confusion matrix (quais erros?), precision/recall
(erros de tipo I vs tipo II), calibracao (o modelo sabe quando esta inseguro?).

### Definicao Formal

- **Top-k Accuracy:** % de vezes que a classe correta esta entre as k predicoes mais provaveis
- **Confusion Matrix:** matriz C onde C[i,j] = quantas amostras da classe i foram preditas como j
- **Precision/Recall/F1:** por classe, para identificar classes problematicas
- **Calibracao:** ECE = sum(|accuracy(bin) - confidence(bin)|) / N

### Por que em ML: Accuracy Sozinha Engana

Num dataset com 95% gatos e 5% caes, um modelo que prediz "gato" sempre tem 95% accuracy
mas e completamente inutil para detectar caes. Metricas por classe revelam a verdade.

In [ ]:
# Simulacao completa de metricas de classificacao
np.random.seed(42)

n_classes = 5
class_names = ['Gato', 'Cao', 'Passaro', 'Peixe', 'Cobra']
n_samples = 500

# Simular predicoes (modelo razoavel mas com confusoes especificas)
true_labels = np.random.choice(n_classes, n_samples, p=[0.35, 0.25, 0.2, 0.12, 0.08])
pred_labels = true_labels.copy()

# Introduzir erros realistas
for i in range(n_samples):
    if np.random.random() < 0.15:  # 15% de erro
        if true_labels[i] == 0:  # Gato confundido com Cao
            pred_labels[i] = 1 if np.random.random() < 0.7 else np.random.randint(n_classes)
        elif true_labels[i] == 4:  # Cobra (classe rara) mais erros
            pred_labels[i] = np.random.randint(n_classes)

# Confusion Matrix
cm = np.zeros((n_classes, n_classes), dtype=int)
for t, p in zip(true_labels, pred_labels):
    cm[t, p] += 1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
ax = axes[0]
im = ax.imshow(cm, cmap='Blues')
ax.set_title('Confusion Matrix', fontweight='bold')
ax.set_xlabel('Predito')
ax.set_ylabel('Verdadeiro')
ax.set_xticks(range(n_classes))
ax.set_yticks(range(n_classes))
ax.set_xticklabels(class_names, rotation=45)
ax.set_yticklabels(class_names)
for i in range(n_classes):
    for j in range(n_classes):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                color='white' if cm[i,j] > cm.max()/2 else 'black', fontsize=11)
plt.colorbar(im, ax=ax)

# Per-class metrics
precisions, recalls, f1s = [], [], []
for i in range(n_classes):
    tp = cm[i, i]
    fp = cm[:, i].sum() - tp
    fn = cm[i, :].sum() - tp
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
    precisions.append(prec)
    recalls.append(rec)
    f1s.append(f1)

ax = axes[1]
x_pos = np.arange(n_classes)
width = 0.25
ax.bar(x_pos - width, precisions, width, label='Precision', color='#3498db')
ax.bar(x_pos, recalls, width, label='Recall', color='#e74c3c')
ax.bar(x_pos + width, f1s, width, label='F1-Score', color='#2ecc71')
ax.set_xlabel('Classe')
ax.set_ylabel('Score')
ax.set_title('Metricas por Classe', fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(class_names)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, 1.1)

plt.tight_layout()
plt.savefig('/tmp/metrics_evaluation.png', dpi=100, bbox_inches='tight')
plt.show()

# Report
overall_acc = np.trace(cm) / cm.sum()
print(f'Overall Accuracy: {overall_acc:.1%}')
print()
print(f'{"Classe":<10} {"Precision":<12} {"Recall":<12} {"F1":<12} {"Suporte":<10}')
print('-' * 56)
for i in range(n_classes):
    support = cm[i, :].sum()
    print(f'{class_names[i]:<10} {precisions[i]:<12.3f} {recalls[i]:<12.3f} '
          f'{f1s[i]:<12.3f} {support:<10}')

### O que observar

1. A confusion matrix mostra que **Gato vs Cao** e a principal confusao (animais similares)
2. **Cobra** (classe rara com apenas 8% dos dados) tem precision/recall mais baixos
3. Accuracy geral (~85%) esconde que a performance em classes raras e significativamente pior
4. F1-Score combina precision e recall -- util para ver o "equilibrio" por classe
5. Classes com alto suporte (Gato: 35%) dominam a accuracy geral

### O que concluir

Sempre avalie modelos de classificacao com **metricas por classe**, especialmente em datasets
desbalanceados. A confusion matrix revela *quais* erros o modelo comete, nao apenas *quantos*.
Para datasets muito desbalanceados, use macro-F1 (media entre classes) em vez de accuracy.

### Conexao com outros notebooks

Em 3_1 (classificacao completa), vimos precision, recall e F1 para classificacao binaria.
Aqui, estendemos para multi-classe. Em 1_5 (design de experimentos), vimos a importancia
de metricas adequadas -- o mesmo principio aplica-se a avaliacao de modelos.

## 5. Problemas Especificos de Classificacao

### Analogia: Diferentes Tipos de Prova

- **Classificacao padrao:** prova de multipla escolha (uma resposta correta)
- **Fine-grained:** prova sobre racas de caes (distincoes sutis)
- **Multi-label:** checklist (multiplas respostas corretas por pergunta)
- **Long-tail:** prova onde 90% das perguntas sao sobre 10% da materia

### Por que em ML: Cada Problema Requer Estrategia Diferente

Fine-grained classification precisa de attention para detalhes sutis.
Multi-label requer BCE em vez de Cross-Entropy. Long-tail precisa de
resampling ou loss rebalanceada. Usar a abordagem errada desperdicaria
tempo e dados.

In [ ]:
# Visualizacao de distribuicoes long-tail e estrategias
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 1. Long-tail distribution
np.random.seed(42)
n_classes_lt = 20
# Distribuicao Zipf-like
counts = np.array([int(5000 / (i+1)**1.2) for i in range(n_classes_lt)])

ax = axes[0]
colors_lt = plt.cm.RdYlGn_r(np.linspace(0, 1, n_classes_lt))
ax.bar(range(n_classes_lt), counts, color=colors_lt, edgecolor='black', alpha=0.8)
ax.set_xlabel('Classe (ordenada por frequencia)')
ax.set_ylabel('Numero de Amostras')
ax.set_title('Distribuicao Long-Tail', fontweight='bold')
ax.axhline(y=np.mean(counts), color='red', linestyle='--', label=f'Media: {np.mean(counts):.0f}')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# 2. Multi-label vs single-label
ax = axes[1]
# Single-label: exatamente um 1
single_label = np.array([1, 0, 0, 0, 0])
# Multi-label: multiplos 1s
multi_label = np.array([1, 0, 1, 0, 1])

x_pos = np.arange(5)
width = 0.35
ax.bar(x_pos - width/2, single_label, width, label='Single-Label', color='#3498db')
ax.bar(x_pos + width/2, multi_label, width, label='Multi-Label', color='#e74c3c')
ax.set_xlabel('Classe')
ax.set_ylabel('Label')
ax.set_title('Single vs Multi-Label', fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(['Gato', 'Cao', 'Jardim', 'Noite', 'Chuva'])
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# 3. Estrategias para long-tail
ax = axes[2]
strategies = ['Nenhuma', 'Class\nWeights', 'Over\nsampling', 'Focal\nLoss', 'Decoupled\nTraining']
accs_head = [95, 93, 91, 92, 94]  # classes frequentes
accs_tail = [40, 65, 72, 70, 75]  # classes raras

x_pos = np.arange(len(strategies))
ax.bar(x_pos - 0.2, accs_head, 0.4, label='Classes Frequentes', color='#3498db')
ax.bar(x_pos + 0.2, accs_tail, 0.4, label='Classes Raras', color='#e74c3c')
ax.set_xlabel('Estrategia')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Long-Tail: Estrategias de Rebalanceamento', fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(strategies)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('/tmp/specific_problems.png', dpi=100, bbox_inches='tight')
plt.show()

print('Long-tail: classe mais frequente tem 50x mais amostras que a mais rara')
print('Multi-label: uma imagem pode ter gato + jardim + chuva simultaneamente')
print('Decoupled Training: melhor trade-off entre classes frequentes e raras')

### O que observar

1. Na distribuicao long-tail, a classe 0 tem **5000** amostras, a classe 19 tem **~100**
2. Sem tratamento, classes raras tem **40% accuracy** vs 95% para frequentes
3. Oversampling e Focal Loss melhoram classes raras mas podem prejudicar frequentes levemente
4. Multi-label permite multiplas classes por imagem -- requer BCE, nao CE
5. Decoupled Training (treinar features e classifier separadamente) e o melhor trade-off

### O que concluir

Problemas especificos de classificacao exigem adaptacoes especificas. O pipeline padrao
(CE + balanced sampling) nao funciona para todos os cenarios. Identificar o tipo de
problema (balanceado, long-tail, multi-label, fine-grained) e o primeiro passo antes
de escolher arquitetura e loss function.

### Conexao com outros notebooks

Long-tail distributions conectam com 1_1 (estatistica descritiva) -- entender a
distribuicao dos dados e fundamental antes de treinar. Multi-label conecta com
3_1 (classificacao), onde vimos classificacao binaria que se generaliza para multi-label.

### O que observar sobre Ensembling e TTA na Pratica

Na pratica profissional, ensembling e TTA sao usados quase exclusivamente em:
1. **Competicoes (Kaggle):** onde 0.1% de accuracy importa
2. **Producao critica:** diagnostico medico, veiculos autonomos
Em projetos normais, o custo computacional geralmente nao compensa o ganho marginal.

### O que concluir sobre a Evolucao dos Training Recipes

Os training recipes modernos (cosine annealing, warmup, label smoothing, mixup) foram
sistematicamente validados em papers como "Bag of Tricks" (He et al., 2019). Cada tecnica
adiciona ~0.5% de accuracy, mas combinadas podem somar 3-5% sem mudar a arquitetura.

### Conexao com outros notebooks sobre Pipelines de Treino

O pipeline completo de classificacao conecta todos os notebooks anteriores:
dados (2_2 EDA), pre-processamento (2_1 Python), arquitetura (5A_1 CNN fundamentos),
treinamento (4_3), avaliacao (3_1 classificacao), e deploy (6_1).

### Por que em ML: Classificacao como Base para Tarefas Mais Complexas

Classificacao de imagens e a base para deteccao (5A_3), segmentacao (5A_4), e Vision
Transformers (5A_5). O backbone pre-treinado em classificacao e reutilizado em todas
essas tarefas. Dominar classificacao e pre-requisito para o resto do modulo.

## 6. Exercicios Praticos

### Exercicio 1: Comparar Cross-Entropy vs Focal Loss

Simule predicoes de um modelo em um dataset desbalanceado e compare
como CE e Focal Loss pesam os erros de classes raras vs frequentes.

In [ ]:
# TAREFA DO ALUNO: Exercicio 1 - Comparar losses em dataset desbalanceado
# Dataset: 90% classe 0 (facil), 10% classe 1 (dificil)
n = 1000
# Simular probabilidades preditas
p_class0 = np.random.beta(8, 2, int(0.9 * n))   # alta confianca
p_class1 = np.random.beta(2, 4, int(0.1 * n))   # baixa confianca

# TAREFA DO ALUNO: Calcular CE loss e Focal loss (gamma=2) para cada grupo
# CE: -log(p)
# Focal: -(1-p)^gamma * log(p)
gamma = 2

# ce_loss_0 = None  # loss media classe 0
# ce_loss_1 = None  # loss media classe 1
# focal_loss_0 = None
# focal_loss_1 = None
ce_loss_0 = None
ce_loss_1 = None
focal_loss_0 = None
focal_loss_1 = None

In [ ]:
# SOLUCAO - Exercicio 1
n = 1000
p_class0 = np.random.beta(8, 2, int(0.9 * n))
p_class1 = np.random.beta(2, 4, int(0.1 * n))
gamma = 2

ce_loss_0 = np.mean(-np.log(p_class0 + 1e-8))
ce_loss_1 = np.mean(-np.log(p_class1 + 1e-8))
focal_loss_0 = np.mean(-(1-p_class0)**gamma * np.log(p_class0 + 1e-8))
focal_loss_1 = np.mean(-(1-p_class1)**gamma * np.log(p_class1 + 1e-8))

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(2)
width = 0.35
ax.bar(x - width/2, [ce_loss_0, ce_loss_1], width, label='Cross-Entropy', color='#3498db')
ax.bar(x + width/2, [focal_loss_0, focal_loss_1], width, label='Focal (gamma=2)', color='#e74c3c')
ax.set_xticks(x)
ax.set_xticklabels(['Classe 0 (90% - facil)', 'Classe 1 (10% - dificil)'])
ax.set_ylabel('Loss Media')
ax.set_title('CE vs Focal Loss por Classe', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('/tmp/loss_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

ratio_ce = ce_loss_1 / ce_loss_0
ratio_focal = focal_loss_1 / focal_loss_0
print(f'CE: loss classe dificil / facil = {ratio_ce:.1f}x')
print(f'Focal: loss classe dificil / facil = {ratio_focal:.1f}x')
print(f'Focal foca {ratio_focal/ratio_ce:.1f}x mais nos exemplos dificeis que CE')

### Exercicio 2: Construir Confusion Matrix e Metricas

Dado um conjunto de predicoes, construa a confusion matrix e calcule
precision, recall e F1 por classe. Identifique a classe mais problematica.

In [ ]:
# TAREFA DO ALUNO: Exercicio 2 - Confusion matrix e metricas
np.random.seed(123)
classes = ['A', 'B', 'C', 'D']
n_samples = 200

# Gerar dados (modelo com tendencia a confundir B com C)
true = np.random.choice(4, n_samples)
pred = true.copy()
for i in range(n_samples):
    if true[i] == 1 and np.random.random() < 0.3:  # B confundido
        pred[i] = 2  # com C
    elif np.random.random() < 0.1:
        pred[i] = np.random.randint(4)

# TAREFA DO ALUNO: Construir confusion matrix
# TAREFA DO ALUNO: Calcular precision, recall, F1 para cada classe
# TAREFA DO ALUNO: Identificar a classe com pior F1
# cm = None
# worst_class = None
cm = None
worst_class = None

In [ ]:
# SOLUCAO - Exercicio 2
np.random.seed(123)
classes = ['A', 'B', 'C', 'D']
n_samples = 200
true = np.random.choice(4, n_samples)
pred = true.copy()
for i in range(n_samples):
    if true[i] == 1 and np.random.random() < 0.3:
        pred[i] = 2
    elif np.random.random() < 0.1:
        pred[i] = np.random.randint(4)

# Confusion matrix
cm = np.zeros((4, 4), dtype=int)
for t, p in zip(true, pred):
    cm[t, p] += 1

# Metricas por classe
print(f'{"Classe":<8} {"Precision":<12} {"Recall":<12} {"F1":<12}')
print('-' * 44)
f1_scores = []
for i in range(4):
    tp = cm[i, i]
    fp = cm[:, i].sum() - tp
    fn = cm[i, :].sum() - tp
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
    f1_scores.append(f1)
    print(f'{classes[i]:<8} {prec:<12.3f} {rec:<12.3f} {f1:<12.3f}')

worst_idx = np.argmin(f1_scores)
worst_class = classes[worst_idx]
print(f'\nPior classe: {worst_class} (F1 = {f1_scores[worst_idx]:.3f})')
print(f'Motivo: B e frequentemente confundido com C (veja cm[1,2] = {cm[1,2]})')

### Exercicio 3: Simular Efeito de Label Smoothing

Compare os gradientes de Cross-Entropy com hard labels vs label smoothing
e demonstre por que smoothing reduz overconfidence.

In [ ]:
# TAREFA DO ALUNO: Exercicio 3 - Label Smoothing efeito
# Para um modelo que prediz [0.95, 0.03, 0.02] para classe 0:
# Calcular loss com hard labels [1, 0, 0]
# Calcular loss com smooth labels [0.9, 0.05, 0.05] (eps=0.1)
# Qual loss e maior? Por que isso ajuda?
pred_probs = np.array([0.95, 0.03, 0.02])
hard_target = np.array([1.0, 0.0, 0.0])
eps = 0.1
n_c = 3

# TAREFA DO ALUNO: Calcular smooth_target e ambas as losses
# smooth_target = None
# loss_hard = None
# loss_smooth = None
smooth_target = None
loss_hard = None
loss_smooth = None

In [ ]:
# SOLUCAO - Exercicio 3
pred_probs = np.array([0.95, 0.03, 0.02])
hard_target = np.array([1.0, 0.0, 0.0])
eps = 0.1
n_c = 3

smooth_target = (1 - eps) * hard_target + eps / n_c
loss_hard = -np.sum(hard_target * np.log(pred_probs + 1e-8))
loss_smooth = -np.sum(smooth_target * np.log(pred_probs + 1e-8))

print(f'Hard target:   {hard_target}')
print(f'Smooth target: {smooth_target}')
print()
print(f'Loss (hard):   {loss_hard:.4f}')
print(f'Loss (smooth): {loss_smooth:.4f}')
print()
print('Label smoothing produz loss MAIOR mesmo com predicao quase correta')
print('Isso forca o modelo a nunca ficar 100% confiante')
print('Resultado: melhor calibracao e generalizacao')

### O que observar sobre o Pipeline Completo de Classificacao

Na pratica, um pipeline de classificacao de imagens envolve 6 etapas:
1. **Coleta e curadoria** dos dados (qualidade > quantidade)
2. **Exploracao** com EDA visual (distribuicao de classes, outliers)
3. **Pre-processamento** (resize, normalize, augmentation)
4. **Treino** com transfer learning + training recipes
5. **Avaliacao** com metricas por classe + confusion matrix
6. **Iteracao** baseada nos erros mais frequentes

### O que concluir sobre a Importancia da Curadoria de Dados

Modelos de classificacao sao tao bons quanto os dados. Dados mal anotados (label noise),
imagens duplicadas, e classes ambiguas prejudicam mais que a escolha de arquitetura.
Na pratica, melhorar os dados frequentemente vale mais que refinar o modelo.

### Conexao com outros notebooks sobre Data Quality

A importancia da curadoria de dados conecta com 2_2 (EDA completa), onde vimos como
explorar dados antes de modelar. Tambem conecta com 1_1 (estatistica descritiva),
onde aprendemos a caracterizar distribuicoes -- essencial para identificar imbalance.

### O que observar sobre Deployment de Modelos de Classificacao

Ao levar um modelo para producao, considere:
1. **Latencia:** tempo por imagem (EfficientNet-B0: ~10ms vs ResNet-152: ~50ms)
2. **Throughput:** imagens por segundo em batch
3. **Calibracao:** a confianca do modelo corresponde a accuracy real?
4. **Edge cases:** imagens fora da distribuicao de treino (OOD detection)

### O que concluir sobre Calibracao e Confianca

Um modelo calibrado com 80% de confianca deveria acertar 80% das vezes. Na pratica,
modelos deep tendem a ser overconfident. Temperature scaling e label smoothing ajudam
a calibrar, permitindo definir thresholds de confianca para decisoes automaticas.

### Conexao com outros notebooks sobre Deploy

O pipeline de classificacao conecta diretamente com 6_1 (deploy de modelos), onde veremos
como servir modelos em producao. Calibracao conecta com 1_3 (estatistica bayesiana),
onde vimos probabilidades condicionais e incerteza.

### Por que em ML: Classificacao de Imagens no Mundo Real

Classificacao de imagens e usada em: triagem medica (raio-X normal vs anormal),
controle de qualidade (peca boa vs defeituosa), agricultura (planta saudavel vs doente),
seguranca (reconhecimento facial), e comercio (busca visual de produtos).

### O que observar sobre Transfer Learning Cross-Domain

Transfer learning funciona melhor quando o dominio fonte (ImageNet) e similar ao alvo.
Para dominios muito diferentes (imagens medicas, satelite, microscopio), considere:
- Pre-treinar em dominio intermediario (ex: RadImageNet para medico)
- Self-supervised pre-training (SimCLR, MoCo, DINO)
- Usar augmentations domain-specific

### O que concluir sobre Self-Supervised Learning

Self-supervised learning (SSL) esta mudando o paradigma: em vez de depender de labels,
o modelo aprende representacoes a partir da estrutura dos dados. DINO e MAE mostram
que SSL pode superar supervised pre-training, especialmente com poucos labels.

### Conexao com outros notebooks sobre Aprendizado Nao-Supervisionado

SSL conecta com 3_5 (clustering) e 3_6 (reducao de dimensionalidade) -- sao todas
formas de aprender representacoes sem labels explicitos. Tambem conecta com 5B_4
(Transformers BERT), onde veremos self-supervised learning para texto.

### Por que em ML: O Futuro do Transfer Learning

O futuro aponta para foundation models multi-modais (CLIP, ALIGN) que aprendem
representacoes visuais a partir de texto, eliminando a necessidade de datasets
anotados manualmente. Isso democratiza a visao computacional.

### O que observar sobre Model Selection na Pratica

A escolha do modelo base para transfer learning depende de varios fatores:
- **ResNet-50:** equilibrio entre accuracy e velocidade, o "default" seguro
- **EfficientNet-B0/B1:** melhor para deploy em dispositivos com recursos limitados
- **ViT-Base:** melhor accuracy com datasets grandes (>100K imagens)
- **ConvNeXt:** performance de Transformer com inferencia eficiente de CNN

### O que concluir sobre Benchmarking de Modelos

Nunca confie em benchmarks publicados sem testar no SEU dataset. Performance em ImageNet
nao garante performance em dominios especificos. Sempre faca um benchmark comparativo
com 3-5 arquiteturas no seu dataset antes de investir tempo em tuning.

### Conexao com outros notebooks sobre Escolha de Modelos

A selecao de modelos conecta com 1_5 (design de experimentos) -- usar validacao cruzada
para comparar modelos de forma estatisticamente rigorosa. Tambem conecta com 4_5
(aceleracao hardware), onde latencia e throughput entram na decisao.

## 7. Erros Comuns e Armadilhas

### Erro 1: Reportar Accuracy em Dataset Desbalanceado
**Sintoma:** 95% accuracy mas modelo nao detecta classe minoritaria
**Causa:** Accuracy dominada pela classe majoritaria
**Solucao:** Sempre reporte precision, recall, F1, e confusion matrix

### Erro 2: Usar Mesmo Split para Validacao e Teste
**Sintoma:** Performance "magicamente" alta no teste
**Causa:** Hiperparametros ajustados no test set (data leakage)
**Solucao:** Separar train/val/test e NUNCA tocar no test ate a avaliacao final

### Erro 3: Augmentation no Conjunto de Teste
**Sintoma:** Metricas de teste instáveis entre execucoes
**Causa:** Transformacoes aleatorias nos dados de avaliacao
**Solucao:** Augmentation APENAS no treino; teste usa apenas resize + normalize

### Erro 4: Fine-tuning com Learning Rate Alta
**Sintoma:** Loss sobe no inicio do treinamento
**Causa:** LR alta destroi pesos pre-treinados
**Solucao:** LR 10-100x menor para fine-tuning (1e-4 vs 1e-2)

### Erro 5: Ignorar Calibracao do Modelo
**Sintoma:** Modelo diz 99% confianca em predicoes erradas
**Causa:** Treinamento sem regularizacao de confianca
**Solucao:** Label smoothing, temperature scaling, ou Platt scaling

## 8. Resumo e Conexoes

### Hierarquia de Conceitos

```
Classificacao de Imagens
|
|-- Transfer Learning
|   |-- Feature Extraction (congelar backbone)
|   |-- Fine-tuning (descongelar parcialmente)
|   |-- Discriminative LR (LRs por camada)
|
|-- Training Recipes
|   |-- Progressive Resizing
|   |-- Test-Time Augmentation (TTA)
|   |-- Model Ensembling
|   |-- Label Smoothing / Mixup / CutMix
|
|-- Loss Functions
|   |-- Cross-Entropy (padrao)
|   |-- Focal Loss (class imbalance)
|   |-- Label Smoothing (overconfidence)
|
|-- Metricas
|   |-- Confusion Matrix
|   |-- Precision / Recall / F1 por classe
|   |-- Top-k Accuracy
|   |-- Calibracao (ECE)
|
|-- Problemas Especificos
    |-- Fine-Grained (distincoes sutis)
    |-- Multi-Label (multiplas classes por imagem)
    |-- Long-Tail (classes muito desbalanceadas)
```

### Tabela de Conexoes

| Conceito | Conexao | Notebook |
|----------|---------|----------|
| Transfer learning | Reusar features pre-treinadas | 4_4 transfer_learning |
| Learning rate scheduling | Convergencia otima | 4_3 treinamento_deep |
| Data augmentation | Regularizacao visual | 5A_1 cnn_fundamentos |
| Focal Loss | Classificacao desbalanceada | 3_1 classificacao_completa |
| Confusion matrix | Avaliacao de classificadores | 3_1 classificacao_completa |
| Progressive resizing | Otimizacao de compute | 4_5 aceleracao_hardware |
| Distribuicoes long-tail | Estatistica descritiva | 1_1 estatistica_descritiva |
| Calibracao | Probabilidade e incerteza | 1_3 estatistica_bayesiana |

### Checklist de Competencias

- [ ] Sei escolher entre Feature Extraction, Fine-tuning e Discriminative LR
- [ ] Implemento training recipes modernos (progressive resizing, TTA, ensembling)
- [ ] Escolho a loss function adequada para o tipo de problema
- [ ] Avalio modelos com metricas por classe, nao apenas accuracy
- [ ] Identifico e trato problemas de class imbalance e long-tail
- [ ] Entendo o trade-off entre complexidade do pipeline e ganho de performance

### Proximos Passos

No proximo notebook (5A_3 -- Deteccao de Objetos), iremos alem da classificacao:
em vez de "esta imagem contem um gato?", responderemos "onde esta o gato na imagem?".
Veremos como o backbone de classificacao e reutilizado como feature extractor para
deteccao.